### Quickstart: Compare runs, choose a model, and deploy it to a REST API

In this quickstart, you will:

- Run a hyperparameter sweep on a training script

- Compare the results of the runs in the MLflow UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform


In [1]:
import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

e:\AI\MlOPS\MLFLOW\mlops_venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
e:\AI\MlOPS\MLFLOW\mlops_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [3]:
## Split the data into training,validation and test sets

train,test=train_test_split(data,test_size=0.25,random_state=42)
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [4]:
train[['quality']].values.ravel()

array([6, 6, 5, ..., 6, 6, 8], shape=(3673,))

In [5]:
train_x=train.drop(['quality'],axis=1).values
train_y=train[['quality']].values.ravel()

## test dataset
test_x=test.drop(['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

## splitting this train data into train and validation

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

signature=infer_signature(train_x,train_y)

In [6]:
np.mean(train_x,axis=0)

array([6.86621852e+00, 2.80377808e-01, 3.32597005e-01, 6.42164738e+00,
       4.55513955e-02, 3.53556841e+01, 1.38792376e+02, 9.94074221e-01,
       3.18919333e+00, 4.88396869e-01, 1.05005673e+01])

In [9]:
train_x.shape[1]

11

In [7]:
### ANN Model

def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    ## Define model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation='relu'),
            keras.layers.Dense(1)

        ]
    )

    ## compile the model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    ## Train the ANN model with lr and momentum params wwith MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),
                  epochs=epochs,
                  batch_size=64)
        
        ## Evaluate the model
        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rmse=eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        ## log the model

        mlflow.tensorflow.log_model(model,"model",signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [8]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [9]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)

}

In [10]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")


2026/03/31 21:20:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/31 21:20:59 INFO mlflow.store.db.utils: Updating database tables
2026/03/31 21:21:00 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 18s 401ms/step - loss: 35.3036 - root_mean_squared_error: 5.9417
27/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 32.0050 - root_mean_squared_error: 5.6553   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 26.7399 - root_mean_squared_error: 5.1711 - val_loss: 19.4942 - val_root_mean_squared_error: 4.4152

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 20.9686 - root_mean_squared_error: 4.5791
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 17.6259 - root_mean_squared_error: 4.1958 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.9415 - root_mean_squared_error: 3.8654 - val_loss: 10.8618 - val_ro

2026/03/31 21:21:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 17s 380ms/step - loss: 40.5028 - root_mean_squared_error: 6.3642
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 36.4533 - root_mean_squared_error: 6.0364   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 33.5674 - root_mean_squared_error: 5.7937 - val_loss: 29.6819 - val_root_mean_squared_error: 5.4481

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 30.2123 - root_mean_squared_error: 5.4966
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 28.6263 - root_mean_squared_error: 5.3500 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 26.8973 - root_mean_squared_error: 5.1863 - val_loss: 23.7196 - val_root_mean_squared_error: 4.8703

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 24.6423 - root_mean_squared_error: 4.9641
46/46 ━

2026/03/31 21:21:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 16s 363ms/step - loss: 37.0515 - root_mean_squared_error: 6.0870
43/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 11.2867 - root_mean_squared_error: 3.1960   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4.5730 - root_mean_squared_error: 2.1384 - val_loss: 1.4402 - val_root_mean_squared_error: 1.2001

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 1.1298 - root_mean_squared_error: 1.0629
26/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1469 - root_mean_squared_error: 1.0704 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1170 - root_mean_squared_error: 1.0569 - val_loss: 1.0451 - val_root_mean_squared_error: 1.0223

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 1.1525 - root_mean_squared_error: 1.0735
43/46 ━━━━━━━━

2026/03/31 21:21:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:44<00:00, 11.06s/trial, best loss: 0.7460677623748779]

2026/03/31 21:21:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.007074394705630996), 'momentum': np.float64(0.9150285550552115)}
Best eval rmse: 0.7460677623748779


In [11]:
## Inferencing

from mlflow.models import validate_serving_input

model_uri = 'runs:/9e0b52639ab44e6a864f5bd2f460fe42/model'

# The logged model does not contain an input_example.
# Manually generate a serving payload to verify your model prior to deployment.
from mlflow.models import convert_input_example_to_serving_input

# Define INPUT_EXAMPLE via assignment with your own input example to the model
# A valid input example is a data instance suitable for pyfunc prediction
serving_payload = convert_input_example_to_serving_input(test_x)

# Validate the serving payload works on the model
validate_serving_input(model_uri, serving_payload)

MlflowException: Run with id=9e0b52639ab44e6a864f5bd2f460fe42 not found

In [18]:
# Load model as a PyFuncModel.
model_uri = 'runs:/9e0b52639ab44e6a864f5bd2f460fe42/model'
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Predict on a Pandas DataFrame.
import pandas as pd
loaded_model.predict(pd.DataFrame(test_x))

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


array([[4.2569876],
       [7.2708473],
       [5.690359 ],
       ...,
       [6.881361 ],
       [6.6991577],
       [4.748927 ]], dtype=float32)

In [19]:
## Register in the model registry
mlflow.register_model(model_uri,"wine-quality")

Successfully registered model 'wine-quality'.
Created version '1' of model 'wine-quality'.


<ModelVersion: aliases=[], creation_timestamp=1727548664831, current_stage='None', description=None, last_updated_timestamp=1727548664831, name='wine-quality', run_id='9e0b52639ab44e6a864f5bd2f460fe42', run_link=None, source='file:///e:/UDemy%20Final/MlFlowStarter/2-DLMLFLOW/mlruns/929139454095764868/9e0b52639ab44e6a864f5bd2f460fe42/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=1>